# Cardiac JEPA — pré-entraînement sur Colab

Entraîne le JEPA latent sur PTB-XL (100 Hz) avec le GPU Colab.

**Avant de lancer** : `Exécution ▸ Modifier le type d'exécution ▸ GPU`.

Ce notebook est *idempotent* : re-lance-le après une déconnexion, il reprend
l'entraînement là où il s'était arrêté (`--resume auto`).

**Une seule chose à renseigner** : `REPO_URL` dans la cellule de configuration.

## 1. Vérifier le GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print("torch", torch.__version__, "| cuda dispo :", torch.cuda.is_available())

## 2. Monter Drive et configurer les chemins

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# >>> RENSEIGNE TON URL DE REPO ICI (pas de chevrons !) <<<
# exemple : "https://github.com/moncompte/cardiac-jepa.git"
REPO_URL = ""

if not REPO_URL:
    raise ValueError(
        "REPO_URL est vide. Mets l'URL de ton repo GitHub dans cette cellule, "
        "par ex. https://github.com/moncompte/cardiac-jepa.git")

DRIVE_ROOT  = Path('/content/drive/MyDrive/cardiac-jepa')   # persiste entre sessions
DATA_DRIVE  = DRIVE_ROOT / 'data'                            # les 2 CSV (petits)
CACHE_DRIVE = DRIVE_ROOT / 'ptbxl_100hz.npy'                 # cache signaux (1,05 Go)
CACHE_LOCAL = Path('/content/ptbxl_100hz.npy')               # copie sur disque local = rapide
RUN_DIR     = DRIVE_ROOT / 'runs' / 'tiny_v1'                # checkpoints + metrics.csv
REPO_DIR    = Path('/content/cardiac-jepa')

RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Drive prêt :", DRIVE_ROOT)

## 3. Cloner / mettre à jour le code

Tu édites en local dans VSCode, tu `git push`, puis tu ré-exécutes cette cellule.

In [ ]:
import subprocess

if REPO_DIR.exists():
    r = subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
                       capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)],
                       capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    raise RuntimeError(
        f"git a échoué (code {r.returncode}). Vérifie REPO_URL, que le repo existe, "
        "et qu'il est public (sinon il faut un token d'accès).")

assert (REPO_DIR / "requirements.txt").exists(), \
    f"{REPO_DIR}/requirements.txt introuvable — le clone a-t-il vraiment réussi ?"

%cd {REPO_DIR}
!pip install -q -r requirements.txt
print("code prêt :", REPO_DIR)

## 4. Données (déposées manuellement sur Drive)

Le cache est construit en local et déposé sur Drive — **rien à télécharger ici**.
Arborescence attendue :

```
MyDrive/cardiac-jepa/
├── ptbxl_100hz.npy        (1,05 Go)  cache des signaux 100 Hz
└── data/
    ├── ptbxl_database.csv (6,6 Mo)
    └── scp_statements.csv (10 Ko)
```

Pourquoi un cache plutôt que les fichiers bruts : `records100` contient **~43 000 petits
fichiers**, atroces à uploader sur Drive *et* atroces à lire depuis un Drive monté.
Un seul fichier contigu, copié sur le disque local de Colab, c'est instantané.

L'ordre des lignes de `ptbxl_database.csv` définit l'index du cache : **ces deux fichiers
doivent aller ensemble.**

In [ ]:
import shutil, time
import numpy as np

required = [CACHE_DRIVE,
            DATA_DRIVE / 'ptbxl_database.csv',
            DATA_DRIVE / 'scp_statements.csv']
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Fichiers manquants sur Drive :\n  " + "\n  ".join(str(m) for m in missing) +
        "\n\nDépose-les exactement à ces emplacements, puis relance cette cellule.")

if not CACHE_LOCAL.exists():
    print("Copie du cache Drive -> disque local (~1 Go, une fois par session)...")
    t0 = time.time()
    shutil.copy(CACHE_DRIVE, CACHE_LOCAL)
    print(f"copié en {time.time()-t0:.0f}s")

# Le code lit ces variables d'environnement (voir jepa/data.py).
os.environ['PTBXL_DATA_DIR'] = str(DATA_DRIVE)   # CSV uniquement
os.environ['PTBXL_CACHE']    = str(CACHE_LOCAL)

a = np.load(CACHE_LOCAL, mmap_mode='r')
assert a.shape[1:] == (1000, 12), f"forme inattendue: {a.shape}"
print(f"cache OK : {a.shape} {a.dtype}")

## 5. Sanity check : les données se chargent

In [ ]:
from jepa.data import PTBXLDataset
ds = PTBXLDataset('pretrain')
x = ds[0]
print(f"pretrain={len(ds)} ECG | un échantillon: {tuple(x.shape)} "
      f"| moyenne={x.mean():.3f} std={x.std():.3f}  (z-normé par dérivation)")

## 6. Entraînement

`--resume auto` reprend depuis `latest.pt` s'il existe. Si Colab te déconnecte,
ré-exécute simplement cette cellule : tu perds au plus une epoch.

In [ ]:
!python -m jepa.train \
    --config jepa/configs/tiny.yaml \
    --out "{RUN_DIR}" \
    --resume auto \
    --workers 2

## 7. Courbes de collapse

Le critère de succès de cette phase. À surveiller :
- `emb_std_ctx` **ne doit pas tendre vers 0**,
- `eff_rank_ctx` doit rester élevé,
- `jepa` doit baisser *progressivement* (pas s'effondrer à ~0 d'emblée),
- `eff_rank_tgt` : point de vigilance connu — la VICReg ne régularise que le contexte.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

m = pd.read_csv(RUN_DIR / 'metrics.csv')
tr, va = m[m.phase == 'train'], m[m.phase == 'val']

fig, ax = plt.subplots(1, 4, figsize=(19, 4))

# LE graphe qui compte : qualité de prédiction, insensible à la difficulté des cibles.
ax[0].plot(va.epoch, va.r2, marker='o', color='#C44E52', label='R²')
ax[0].axhline(0, ls='--', c='k', lw=1, label='prédicteur trivial (moyenne)')
ax[0].set_title('R² — apprend-on vraiment ?'); ax[0].set_xlabel('epoch'); ax[0].legend()

ax[1].plot(tr.step, tr.jepa, lw=0.8)
ax[1].set_title('L_jepa (cible mouvante : non comparable seule)'); ax[1].set_xlabel('step')

ax[2].plot(va.epoch, va.emb_std_ctx, label='contexte')
ax[2].plot(va.epoch, va.emb_std_tgt, label='cible')
ax[2].plot(va.epoch, va.pred_std, label='prédiction')
ax[2].axhline(0.1, ls='--', c='r', lw=1, label='seuil collapse')
ax[2].set_title('écart-type des descriptions'); ax[2].set_xlabel('epoch'); ax[2].legend()

ax[3].plot(va.epoch, va.eff_rank_ctx, label='contexte')
ax[3].plot(va.epoch, va.eff_rank_tgt, label='cible')
ax[3].set_title('rang effectif (max 192)'); ax[3].set_xlabel('epoch'); ax[3].legend()

plt.tight_layout(); plt.show()
print(va[['epoch','r2','cos','emb_std_ctx','eff_rank_ctx','eff_rank_tgt']].tail(10).to_string(index=False))

## 8. Sonde linéaire — la vraie mesure de qualité

Le R² est un diagnostic, pas l'objectif. Seule la sonde dit si les représentations
valent quelque chose : encodeur cible EMA **gelé** → moyenne des 480 tokens → 192 features
→ une couche linéaire → 5 superclasses. Métrique : **macro-AUROC sur le fold 10** (jamais
utilisé jusqu'ici).

**On lance d'abord la baseline de contrôle** : le même protocole sur un encodeur aux poids
**aléatoires**. Si le JEPA ne la bat pas nettement, il n'a rien appris d'utile.

In [ ]:
# Baseline de contrôle : encodeur NON entraîné.
!python -m jepa.probe --random-init --workers 2 --out "{RUN_DIR}/probe_random.json"

In [ ]:
# Sonde sur le JEPA pré-entraîné (dernier checkpoint : 100 epochs -> ckpt_e99.pt).
CKPT = RUN_DIR / 'ckpt_e99.pt'
assert CKPT.exists(), f"{CKPT} absent — liste: {sorted(p.name for p in RUN_DIR.glob('*.pt'))}"
!python -m jepa.probe --ckpt "{CKPT}" --encoder target --workers 2 --out "{RUN_DIR}/probe_jepa.json"

## 9. Ablation `lambda_cov`

Hypothèse : la pénalité de covariance blanchit trop les représentations (rang 175/192)
et plafonne la prédictibilité. On relance à l'identique avec `lambda_cov: 0.04 → 0.005`.
**Une seule variable change.**

On compare ensuite les **AUROC**, pas les R².

In [ ]:
RUN_ABL = DRIVE_ROOT / 'runs' / 'tiny_lowcov'
RUN_ABL.mkdir(parents=True, exist_ok=True)

!python -m jepa.train \
    --config jepa/configs/tiny_lowcov.yaml \
    --out "{RUN_ABL}" \
    --resume auto \
    --workers 2

In [ ]:
CKPT_ABL = RUN_ABL / 'ckpt_e99.pt'
assert CKPT_ABL.exists(), f"{CKPT_ABL} absent"
!python -m jepa.probe --ckpt "{CKPT_ABL}" --encoder target --workers 2 --out "{RUN_ABL}/probe_jepa.json"

### Comparatif final

In [ ]:
import json, pandas as pd

rows = []
for name, path in [("random (contrôle)", RUN_DIR/'probe_random.json'),
                   ("JEPA lambda_cov=0.04", RUN_DIR/'probe_jepa.json'),
                   ("JEPA lambda_cov=0.005", RUN_ABL/'probe_jepa.json')]:
    if path.exists():
        r = json.load(open(path))
        rows.append({"modèle": name, "macro-AUROC (test)": round(r["test_macro_auroc"], 4),
                     **{k: round(v, 3) for k, v in r["test_per_class"].items()}})
print(pd.DataFrame(rows).to_string(index=False))
print("\nRéférence : un ResNet supervisé atteint ~0.93 de macro-AUROC sur ce benchmark.")